# ATR — Early-error detection and recovery pilot

**Selected idea:** Independent state-consistency verifier  
**Selected source set:** Recent failure detection and recovery research  
**Approved verifier policy:** The verifier reports a mismatch only. The LLM must decide how to repair its own belief state.

This notebook is a setup and pilot scaffold. It is not a completed experiment. Run it in Google Colab, inspect the logs, and fix any issues before the confirmatory experiment. Do not report pilot results as final evidence.

## A. Finalized experimental protocol

### Research question
Does an independent pre-action state-consistency verifier improve recovery from deliberately injected early belief-state errors compared with a reflection-based agent, and does its effect vary with task horizon?

### Conditions
- **C1 — Reflection-only baseline:** The agent proposes an action from its belief state. The environment executes or rejects it and returns ordinary execution feedback. The agent may reflect and update its belief.
- **C2 — Verifier report + reflection:** Before execution, a separate Python verifier compares the agent's belief with the authoritative current state. If it finds a mismatch, it reports the mismatched facts to the LLM. **The verifier does not modify the belief.** The LLM must generate the belief correction.

### Controls
Use the same model, task instances, prompts, maximum model calls, action budget, retry budget and decoding settings in both conditions. The verifier may inspect only the current authoritative state, current belief and proposed action. It must not receive future task requirements, hidden fault labels or future answers.

### Environment and horizon
A deterministic research-workflow simulator with meaningful dependencies. The initial pilot uses a 4-action workflow. The planned full study uses 4-, 8- and 12-action workflows.

### Fault injection
After the first two successful actions, set one agent belief fact to an incorrect value while leaving the authoritative environment unchanged. Pilot fault: set `dataset_validated=True` when the true value is `False`.

### Pilot
Run one clean and one injected-error episode for C1 and C2 at horizon 4 (four episodes). This validates mechanics only; it cannot establish efficacy or novelty.

### Planned main evaluation
Initial target: 10 matched task instances × 3 horizons × 2 conditions × 2 trial types (clean/injected) = 120 episodes. This is a feasibility target, not a statistically powered sample-size claim. Reassess after the pilot.

### Outcomes and decision rules
- **Primary:** injected-error recovery rate; success means all required actions complete and the authoritative final state satisfies the task.
- **Secondary:** invalid actions after injection, verifier detection, false-positive rate on clean runs, model calls, generated tokens and elapsed time.
- **Horizon analysis:** compare C2−C1 recovery-rate differences at each horizon and examine the condition × horizon interaction.
- Proposed practical targets: at least +10 percentage points recovery improvement, clean-state false-positive rate no more than 5%, and median runtime overhead no more than 20%. These are pre-study engineering thresholds, not literature findings. Revisit only after the pilot and freeze before main runs.
- Report raw counts and paired uncertainty intervals. If the sample is too small or intervals are wide, report the result as inconclusive rather than claiming success.

### Scope
This is a controlled synthetic research-workflow experiment with one model family and a defined fault type. It cannot establish that the method generalizes to all agents or real autonomous scientific research.

## B. Runtime setup

Use a fresh Google Colab runtime. A GPU is helpful but availability is not guaranteed. If the model cannot load or generation is too slow, record the limitation and do not silently change models between conditions.

In [ ]:
!pip -q install "transformers>=4.45,<5" accelerate sentencepiece

In [ ]:
import json, re, time, random, platform
from copy import deepcopy
from collections import Counter

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## C. Load the local open-weight model

Model choice for the pilot: `Qwen/Qwen2.5-1.5B-Instruct`. Record the exact model revision and runtime in your experiment log. No paid API is required.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()

print("Loaded:", MODEL_ID)
print("Device:", model.get_input_embeddings().weight.device)

## D. Model helper and strict JSON parsing

Malformed output is recorded as a format error. Do not silently repair malformed JSON because that would hide model reliability problems.

In [ ]:
def ask_model(system_text, user_text, max_new_tokens=180):
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text}
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    start = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    elapsed = time.time() - start
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return raw, elapsed, int(new_tokens.shape[0])


def parse_json_object(raw):
    match = re.search(r"\{.*\}", raw, flags=re.DOTALL)
    if not match:
        return None, "No JSON object found"
    try:
        obj = json.loads(match.group(0))
    except json.JSONDecodeError as exc:
        return None, f"Invalid JSON: {exc}"
    if not isinstance(obj, dict):
        return None, "JSON root must be an object"
    return obj, None

## E. Define the deterministic research-workflow environment

The simulator's authoritative state is separate from the agent's belief state. The environment—not the LLM—determines whether actions are valid.

In [ ]:
ACTION_RULES = {
    "define_question": {"requires": [], "sets": ["question_defined"]},
    "select_dataset": {"requires": ["question_defined"], "sets": ["dataset_selected"]},
    "validate_dataset": {"requires": ["dataset_selected"], "sets": ["dataset_validated"]},
    "select_method": {"requires": ["question_defined", "dataset_validated"], "sets": ["method_selected"]},
    "select_metric": {"requires": ["method_selected"], "sets": ["metric_selected"]},
    "run_experiment": {"requires": ["method_selected", "metric_selected", "dataset_validated"], "sets": ["experiment_run"]},
    "validate_results": {"requires": ["experiment_run"], "sets": ["results_validated"]},
    "draft_report": {"requires": ["results_validated"], "sets": ["report_drafted"]},
    "check_reproducibility": {"requires": ["report_drafted"], "sets": ["reproducibility_checked"]},
    "archive_artifacts": {"requires": ["reproducibility_checked"], "sets": ["artifacts_archived"]},
    "peer_review": {"requires": ["artifacts_archived"], "sets": ["peer_reviewed"]},
    "finalize_submission": {"requires": ["peer_reviewed"], "sets": ["submission_finalized"]}
}

HORIZON_ACTIONS = {
    4: ["define_question", "select_dataset", "validate_dataset", "select_method"],
    8: ["define_question", "select_dataset", "validate_dataset", "select_method",
        "select_metric", "run_experiment", "validate_results", "draft_report"],
    12: ["define_question", "select_dataset", "validate_dataset", "select_method",
         "select_metric", "run_experiment", "validate_results", "draft_report",
         "check_reproducibility", "archive_artifacts", "peer_review", "finalize_submission"]
}

FACTS = sorted({
    fact for rule in ACTION_RULES.values()
    for fact in rule["requires"] + rule["sets"]
})


class ResearchWorkflow:
    def __init__(self, horizon=4):
        if horizon not in HORIZON_ACTIONS:
            raise ValueError("Horizon must be 4, 8 or 12")
        self.horizon = horizon
        self.required = HORIZON_ACTIONS[horizon]
        self.true_state = {fact: False for fact in FACTS}
        self.completed = []
        self.invalid_actions = []
        self.events = []

    def execute(self, action):
        if action not in self.required:
            self.invalid_actions.append(action)
            return False, "Action is not part of this task."
        missing = [
            fact for fact in ACTION_RULES[action]["requires"]
            if not self.true_state[fact]
        ]
        if missing:
            self.invalid_actions.append(action)
            feedback = "Preconditions not satisfied: " + ", ".join(missing)
            self.events.append({"type": "rejected", "action": action, "feedback": feedback})
            return False, feedback

        for fact in ACTION_RULES[action]["sets"]:
            self.true_state[fact] = True
        if action not in self.completed:
            self.completed.append(action)
        self.events.append({"type": "executed", "action": action})
        return True, "Action executed."

    def done(self):
        return len(self.completed) == len(self.required)

## F. Simulator unit checks — run before loading the LLM into the experiment

These checks confirm that the environment enforces dependencies and that the fault injection does not alter the authoritative state.

In [ ]:
test_env = ResearchWorkflow(4)
ok1, _ = test_env.execute("define_question")
ok2, _ = test_env.execute("select_dataset")
assert ok1 and ok2
assert test_env.true_state["dataset_validated"] is False

test_belief = deepcopy(test_env.true_state)
test_belief["dataset_validated"] = True
assert test_env.true_state["dataset_validated"] is False
assert test_belief["dataset_validated"] is True

ok3, feedback3 = test_env.execute("select_method")
assert not ok3 and "dataset_validated" in feedback3

print("Environment and fault-isolation checks passed.")

## G. Agent prompts and belief repair

The agent must return JSON with:
- `action`: one allowed action, or `null` if it first needs to repair its belief.
- `belief_updates`: a dictionary of fact names and Boolean values. It may be empty.

When C2's verifier reports a mismatch, the LLM receives the mismatch evidence and decides which belief update to make. The Python verifier does not write to the belief state.

In [ ]:
SYSTEM_PROMPT = """
You are an agent executing a research workflow.
Return exactly one JSON object with keys:
'action' and 'belief_updates'.
'action' must be one allowed action or null if you need to repair your belief first.
'belief_updates' must be an object mapping fact names to true or false.
Only update facts when supported by the supplied observation or verifier report.
Do not invent actions or facts.
"""


def agent_prompt(env, belief, feedback, allowed_actions):
    return (
        f"Task horizon: {env.horizon} required actions.\\n"
        f"Completed actions: {env.completed}\\n"
        f"Your belief state: {json.dumps(belief)}\\n"
        f"Allowed actions according to your belief: {allowed_actions}\\n"
        f"Latest observation/verifier report: {feedback or 'None'}\\n"
        "If the observation shows a belief is wrong, update it in belief_updates. "
        "Then choose a valid next action. Return JSON only."
    )


def apply_agent_updates(belief, updates):
    if not isinstance(updates, dict):
        return belief, ["belief_updates was not an object"]
    errors = []
    for key, value in updates.items():
        if key not in FACTS or not isinstance(value, bool):
            errors.append(f"Rejected invalid belief update: {key}={value}")
            continue
        belief[key] = value
    return belief, errors


def allowed_from_belief(env, belief):
    allowed = []
    for action in env.required:
        if action in env.completed:
            continue
        if all(belief.get(fact, False) for fact in ACTION_RULES[action]["requires"]):
            allowed.append(action)
    return allowed

## H. Verifier: report-only policy

The verifier compares current belief facts with authoritative current facts. It returns a report but never changes the belief. This is the approved A design.

In [ ]:
def verify_state_consistency(true_state, belief_state):
    mismatches = {
        fact: {"agent_belief": belief_state.get(fact), "authoritative_state": true_state[fact]}
        for fact in true_state
        if belief_state.get(fact) != true_state[fact]
    }
    if mismatches:
        return {
            "mismatch_found": True,
            "report": "Current-state mismatch detected. Review the following facts and update your belief only if justified.",
            "mismatches": mismatches
        }
    return {"mismatch_found": False, "report": "No current-state mismatch detected.", "mismatches": {}}


# Unit test: verifier reports, but does not repair.
true_test = {"dataset_validated": False}
belief_test = {"dataset_validated": True}
report_test = verify_state_consistency(true_test, belief_test)
assert report_test["mismatch_found"] is True
assert belief_test["dataset_validated"] is True
print("Report-only verifier check passed:", report_test)

## I. Pilot episode runner

Both conditions have the same maximum number of model calls. C1 receives ordinary environment feedback after execution. C2 receives a verifier report before execution. In both conditions, only the LLM can update its belief.

In [ ]:
def run_episode(horizon=4, condition="C1", inject_error=False, max_model_calls=24):
    env = ResearchWorkflow(horizon)
    belief = deepcopy(env.true_state)
    pending_feedback = ""
    injected = False
    verifier_detected_injected_error = False
    false_positive = False
    calls = 0
    tokens = 0
    seconds = 0.0
    format_errors = 0
    log = []

    while not env.done() and calls < max_model_calls:
        # Inject a single incorrect belief after two successful actions.
        if inject_error and not injected and len(env.completed) >= 2:
            belief["dataset_validated"] = True
            injected = True
            log.append({
                "type": "fault_injected",
                "fact": "dataset_validated",
                "belief": True,
                "authoritative": env.true_state["dataset_validated"]
            })

        # C2 checks before proposing/executing the next action.
        if condition == "C2":
            report = verify_state_consistency(env.true_state, belief)
            if report["mismatch_found"]:
                if not injected:
                    false_positive = True
                if injected:
                    verifier_detected_injected_error = True
                pending_feedback = json.dumps(report)
                log.append({"type": "verifier_report", **report})

        allowed = allowed_from_belief(env, belief)
        prompt = agent_prompt(env, belief, pending_feedback, allowed)
        raw, elapsed, new_tokens = ask_model(SYSTEM_PROMPT, prompt)
        calls += 1
        seconds += elapsed
        tokens += new_tokens

        obj, parse_error = parse_json_object(raw)
        if parse_error:
            format_errors += 1
            pending_feedback = "Your previous response was not valid JSON: " + parse_error
            log.append({"type": "format_error", "raw": raw, "error": parse_error})
            continue

        updates = obj.get("belief_updates", {})
        belief, update_errors = apply_agent_updates(belief, updates)
        if updates or update_errors:
            log.append({"type": "agent_belief_update", "updates": updates, "errors": update_errors})

        action = obj.get("action")
        if action is None:
            pending_feedback = "Belief update received. Now choose an allowed next action."
            continue

        # Recompute legal actions after the LLM's belief updates.
        allowed_after_updates = allowed_from_belief(env, belief)
        if action not in allowed_after_updates:
            pending_feedback = "Action not allowed by the updated belief-state preconditions. Choose an allowed action."
            log.append({
                "type": "invalid_proposed_action",
                "action": action,
                "allowed_after_updates": allowed_after_updates
            })
            continue

        ok, feedback = env.execute(action)
        log.append({"type": "action", "action": action, "success": ok, "feedback": feedback})

        if ok:
            # Agent observes the successful action's public effect.
            for fact in ACTION_RULES[action]["sets"]:
                belief[fact] = True
            pending_feedback = "Observed: action executed successfully."
        else:
            # C1's ordinary environment feedback; C2 also receives it after execution.
            pending_feedback = "Environment feedback: " + feedback

    return {
        "condition": condition,
        "horizon": horizon,
        "injected_error_requested": inject_error,
        "fault_injected": injected,
        "verifier_detected_injected_error": verifier_detected_injected_error,
        "false_positive": false_positive,
        "success": env.done(),
        "completed_actions": env.completed,
        "invalid_environment_actions": len(env.invalid_actions),
        "model_calls": calls,
        "generated_tokens": tokens,
        "model_seconds": round(seconds, 2),
        "format_errors": format_errors,
        "log": log
    }

## J. Run the four-episode pilot

This may take time, especially on CPU. If generation is slow, do not change the model for only one condition. Record the runtime limitation and decide whether a smaller pilot is necessary.

In [ ]:
pilot_results = []
for condition in ["C1", "C2"]:
    for inject in [False, True]:
        print("\\nRunning", condition, "injected_error =", inject)
        result = run_episode(
            horizon=4,
            condition=condition,
            inject_error=inject,
            max_model_calls=24
        )
        pilot_results.append(result)
        print({
            "success": result["success"],
            "fault_injected": result["fault_injected"],
            "verifier_detected": result["verifier_detected_injected_error"],
            "false_positive": result["false_positive"],
            "invalid_actions": result["invalid_environment_actions"],
            "model_calls": result["model_calls"],
            "format_errors": result["format_errors"],
            "completed": result["completed_actions"]
        })

## K. Inspect and save pilot logs

Inspect the full logs to confirm the causal sequence: fault injection → (C2 only) verifier report → LLM-generated belief update → action selection → environment outcome.

In [ ]:
for result in pilot_results:
    print("\\n###", result["condition"], "injected =", result["injected_error_requested"])
    for event in result["log"]:
        print(event)

with open("ATR_pilot_results.json", "w") as f:
    json.dump(pilot_results, f, indent=2)

print("Saved ATR_pilot_results.json")

## L. Pilot go/no-go checklist

Do not begin the main evaluation until these are checked:

- [ ] Model loads and the exact model ID/revision is recorded.
- [ ] At least 9 of 10 standalone format-test outputs produce valid JSON with permitted keys and action names.
- [ ] Simulator preconditions/effects pass unit tests independently of the LLM.
- [ ] Fault injection changes the belief only; authoritative state remains unchanged.
- [ ] C2 verifier reports the injected mismatch before action execution.
- [ ] Verifier code does not write to or repair the agent belief.
- [ ] The LLM itself emits the correction in `belief_updates`.
- [ ] C1 receives no verifier report.
- [ ] Clean C2 runs do not produce false-positive reports.
- [ ] Both conditions use the same maximum model-call and action budgets.
- [ ] The logs show every state change and action decision.
- [ ] The model can complete enough clean tasks for the recovery comparison to be meaningful.

If the LLM cannot repair its belief from the report, that is a meaningful pilot failure. Debug prompt clarity and output format before changing the research question. If the 1.5B model is consistently unable to follow the task, report that limitation and test another model only by rerunning both conditions with that same fixed model.

## M. Reproducibility record

Fill this in before the main experiment and keep it with the paper:

- Date/runtime:
- Model ID and exact revision:
- Transformers/PyTorch versions:
- GPU/CPU:
- Prompt version:
- Simulator version / code commit:
- Task seeds:
- Maximum model calls:
- Maximum action attempts:
- Fault type and injection position:
- Any deviations from protocol: